# KFP Pipeline: Fraud Detection with TrainerV2 + Feast + KServe

This notebook builds and submits a **Kubeflow Pipeline** that orchestrates the full MLOps workflow:

| Step | Component | What it does |
|------|-----------|-------------|
| 1 | **Train** | Submits a TrainerV2 TrainJob → fetches from Feast, trains PyTorch model, uploads sklearn model to MinIO |
| 2 | **Quality Gate** | Reads `metrics.json` from MinIO, enforces minimum AUC threshold |
| 3 | **Deploy** | Creates/patches a KServe InferenceService (conditional on quality gate) |

### How it works

The **Train** component runs as a KFP pod (in `kubeflow` namespace). It submits a TrainerV2 TrainJob
which spawns a separate training pod. That training pod:
- Connects to your Feast servers via cross-namespace DNS
- Fetches historical features from the offline store
- Trains a PyTorch `FraudMLP` model
- Exports an sklearn-compatible `model.joblib`
- Uploads everything to MinIO

### Key fix: TrainingRuntime namespace workaround

The Kubeflow SDK (`get_runtime()`) only looks for namespace-scoped `TrainingRuntime` resources,
but `torch-distributed` is a cluster-scoped `ClusterTrainingRuntime`. The Train component
automatically creates a namespace-scoped copy before submitting, so it works from any namespace.

## 1. Install dependencies

In [1]:
!pip install -q kfp kubeflow


[notice] A new release of pip is available: 24.2 -> 26.0.1
[notice] To update, run: pip install --upgrade pip


## 2. Import KFP

In [2]:
from kfp import dsl, compiler
print(f'KFP imported successfully')

KFP imported successfully


---
## 3. Pipeline Components

### Component 1 — Submit TrainerV2 TrainJob

This component:
1. Ensures a namespace-scoped `TrainingRuntime` exists (copies from `ClusterTrainingRuntime` if needed)
2. Submits a TrainJob via `TrainerClient`
3. Waits for it to complete and streams logs

The training function (defined inline) is the same one validated in `run_training_feast_minio.py`:
Feast fetch → PyTorch train → sklearn export → MinIO upload.

In [3]:
@dsl.component(
    base_image='registry.access.redhat.com/ubi9/python-311:latest',
    packages_to_install=['kubeflow', 'kubernetes'],
)
def submit_train_job(
    workshop_ns: str,
    feast_start_date: str,
    feast_end_date: str,
    num_epochs: int,
    batch_size: int,
    learning_rate: float,
    hidden_dim: int,
    val_split: float,
    minio_endpoint: str,
    minio_access_key: str,
    minio_secret_key: str,
    minio_bucket: str,
    minio_model_prefix: str,
):
    """Submit a TrainerV2 TrainJob that fetches from Feast, trains, uploads to MinIO."""

    # ── Step 1: Ensure TrainingRuntime exists in this namespace ──────────
    # The SDK's get_runtime() only checks namespace-scoped TrainingRuntime.
    # torch-distributed is a ClusterTrainingRuntime. We create a local copy.
    from kubernetes import client, config

    config.load_incluster_config()
    k8s = client.CustomObjectsApi()
    current_ns = open('/var/run/secrets/kubernetes.io/serviceaccount/namespace').read().strip()

    try:
        k8s.get_namespaced_custom_object(
            'trainer.kubeflow.org', 'v1alpha1',
            current_ns, 'trainingruntimes', 'torch-distributed',
        )
        print(f'TrainingRuntime torch-distributed found in {current_ns}')
    except client.ApiException as e:
        if e.status == 404:
            ctr = k8s.get_cluster_custom_object(
                'trainer.kubeflow.org', 'v1alpha1',
                'clustertrainingruntimes', 'torch-distributed',
            )
            tr = {
                'apiVersion': 'trainer.kubeflow.org/v1alpha1',
                'kind': 'TrainingRuntime',
                'metadata': {
                    'name': 'torch-distributed',
                    'namespace': current_ns,
                    'labels': ctr['metadata'].get('labels', {}),
                },
                'spec': ctr['spec'],
            }
            k8s.create_namespaced_custom_object(
                'trainer.kubeflow.org', 'v1alpha1',
                current_ns, 'trainingruntimes', tr,
            )
            print(f'Created TrainingRuntime torch-distributed in {current_ns}')
        else:
            raise

    # ── Step 2: Define the training function ────────────────────────────
    # This function runs inside the TrainerV2 training pod (not this pod).
    # It must be self-contained — all imports happen inside.
    def train_fraud_from_feast(
        num_epochs=5, batch_size=256, lr=1e-3, hidden_dim=64,
        feast_registry_url=None, feast_offline_host=None, feast_offline_port=8815,
        feast_start_date='2025-01-01', feast_end_date='2025-03-31',
        feast_features=None, label_column='fraud', val_split=0.2,
        output_dir='/tmp/model_output',
        minio_endpoint='http://minio-service.kubeflow.svc.cluster.local:9000',
        minio_access_key='minio', minio_secret_key='minio123',
        minio_bucket='models', minio_model_prefix='fraud-detector',
        workshop_ns=None,
    ):
        import json, os, random
        import numpy as np
        import pandas as pd
        import torch
        import torch.distributed as dist
        from sklearn.metrics import (
            roc_auc_score, accuracy_score, precision_score,
            recall_score, f1_score, confusion_matrix,
        )
        from sklearn.model_selection import train_test_split
        from torch import nn
        from torch.utils.data import DataLoader, Dataset, DistributedSampler

        # Kubeflow SDK passes func_args as a single dict on K8s backend
        if isinstance(num_epochs, dict):
            cfg = num_epochs
            num_epochs = cfg.get('num_epochs', 5)
            batch_size = cfg.get('batch_size', 256)
            lr = cfg.get('lr', 1e-3)
            hidden_dim = cfg.get('hidden_dim', 64)
            feast_registry_url = cfg.get('feast_registry_url')
            feast_offline_host = cfg.get('feast_offline_host')
            feast_offline_port = cfg.get('feast_offline_port', 8815)
            feast_start_date = cfg.get('feast_start_date', feast_start_date)
            feast_end_date = cfg.get('feast_end_date', feast_end_date)
            feast_features = cfg.get('feast_features')
            label_column = cfg.get('label_column', 'fraud')
            val_split = cfg.get('val_split', 0.2)
            output_dir = cfg.get('output_dir', output_dir)
            minio_endpoint = cfg.get('minio_endpoint', minio_endpoint)
            minio_access_key = cfg.get('minio_access_key', minio_access_key)
            minio_secret_key = cfg.get('minio_secret_key', minio_secret_key)
            minio_bucket = cfg.get('minio_bucket', minio_bucket)
            minio_model_prefix = cfg.get('minio_model_prefix', minio_model_prefix)
            workshop_ns = cfg.get('workshop_ns', workshop_ns)

        ns = workshop_ns or os.environ.get('WORKSHOP_NS', 'mlops-workshop')
        if feast_registry_url is None:
            feast_registry_url = f'feast-registry-service.{ns}.svc.cluster.local:6567'
        if feast_offline_host is None:
            feast_offline_host = f'feast-offline-service.{ns}.svc.cluster.local'
        if minio_model_prefix == 'fraud-detector':
            minio_model_prefix = f'fraud-detector-{ns}'

        random.seed(42); np.random.seed(42); torch.manual_seed(42)
        world_size = int(os.getenv('WORLD_SIZE', '1'))
        rank = int(os.getenv('RANK', '0'))
        distributed = world_size > 1
        if distributed:
            dist.init_process_group(backend='nccl' if torch.cuda.is_available() else 'gloo')

        # ── Feast fetch ──
        if rank == 0:
            print(f'Namespace: {ns}')
            print(f'Fetching from Feast ({feast_start_date} to {feast_end_date})...')

        from datetime import datetime
        from feast import FeatureStore, RepoConfig

        if feast_features is None:
            feast_features = [
                'fraud_features:distance_from_home',
                'fraud_features:distance_from_last_transaction',
                'fraud_features:ratio_to_median_purchase_price',
                'fraud_features:repeat_retailer',
                'fraud_features:used_chip',
                'fraud_features:used_pin_number',
                'fraud_features:online_order',
                'fraud_features:fraud',
            ]

        feast_config = RepoConfig(
            project='fraud_detection', provider='local',
            registry={'registry_type': 'remote', 'path': feast_registry_url},
            offline_store={'type': 'remote', 'host': feast_offline_host, 'port': int(feast_offline_port)},
            online_store={'type': 'sqlite', 'path': '/tmp/feast_online.db'},
            entity_key_serialization_version=3,
        )
        store = FeatureStore(config=feast_config)
        df = store.get_historical_features(
            entity_df=None, features=feast_features,
            start_date=datetime.strptime(feast_start_date, '%Y-%m-%d'),
            end_date=datetime.strptime(feast_end_date, '%Y-%m-%d'),
        ).to_df()

        all_cols = [f.split(':')[-1] for f in feast_features]
        feature_cols = [c for c in all_cols if c != label_column]
        df = df.dropna(subset=feature_cols + [label_column])
        train_df, val_df = train_test_split(
            df, test_size=val_split, random_state=42, stratify=df[label_column],
        )
        if rank == 0:
            print(f'Data: {len(df)} rows | Train: {len(train_df)} | Val: {len(val_df)}')

        # ── Model definition ──
        class TabDS(Dataset):
            def __init__(s, f, fc, lc):
                s.x = torch.tensor(f[fc].values, dtype=torch.float32)
                s.y = torch.tensor(f[lc].values, dtype=torch.float32).unsqueeze(1)
            def __len__(s): return len(s.x)
            def __getitem__(s, i): return s.x[i], s.y[i]

        class FraudMLP(nn.Module):
            def __init__(s, d, h):
                super().__init__()
                s.net = nn.Sequential(
                    nn.Linear(d, h), nn.ReLU(), nn.Dropout(0.2),
                    nn.Linear(h, h // 2), nn.ReLU(),
                    nn.Linear(h // 2, 1),
                )
            def forward(s, x): return s.net(x)

        device = torch.device('cpu')
        model = FraudMLP(len(feature_cols), hidden_dim).to(device)
        if distributed:
            model = nn.parallel.DistributedDataParallel(model)

        ts = DistributedSampler(TabDS(train_df, feature_cols, label_column)) if distributed else None
        tl = DataLoader(TabDS(train_df, feature_cols, label_column), batch_size=batch_size, sampler=ts, shuffle=(ts is None))
        vl = DataLoader(TabDS(val_df, feature_cols, label_column), batch_size=batch_size)
        crit = nn.BCEWithLogitsLoss()
        opt = torch.optim.Adam(model.parameters(), lr=lr)

        # ── Training loop ──
        for ep in range(num_epochs):
            model.train()
            if ts: ts.set_epoch(ep)
            rl = 0
            for xb, yb in tl:
                opt.zero_grad()
                loss = crit(model(xb), yb)
                loss.backward()
                opt.step()
                rl += loss.item()

            model.eval()
            tg, sc = [], []
            with torch.no_grad():
                for xb, yb in vl:
                    sc.extend(torch.sigmoid(model(xb)).cpu().numpy().reshape(-1).tolist())
                    tg.extend(yb.numpy().reshape(-1).tolist())
            auc = roc_auc_score(tg, sc) if len(set(int(v) for v in tg)) >= 2 else float('nan')
            if rank == 0:
                print(f'Epoch {ep+1}/{num_epochs} | loss={rl/max(len(tl),1):.4f} | val_auc={auc:.4f}')

        # ── Export + Upload (rank 0 only) ──
        if rank == 0:
            os.makedirs(output_dir, exist_ok=True)
            m = model.module if hasattr(model, 'module') else model

            pt_path = os.path.join(output_dir, 'fraud_mlp_state_dict.pt')
            torch.save({'model_state_dict': m.state_dict(), 'feature_columns': feature_cols, 'hidden_dim': hidden_dim}, pt_path)

            import joblib
            from sklearn.neural_network import MLPClassifier
            clf = MLPClassifier(hidden_layer_sizes=(hidden_dim, hidden_dim // 2), activation='relu', max_iter=1)
            clf.fit(np.zeros((2, len(feature_cols))), np.array([0, 1]))
            m.eval()
            clf.coefs_ = [m.net[0].weight.detach().cpu().numpy().T, m.net[3].weight.detach().cpu().numpy().T, m.net[5].weight.detach().cpu().numpy().T]
            clf.intercepts_ = [m.net[0].bias.detach().cpu().numpy(), m.net[3].bias.detach().cpu().numpy(), m.net[5].bias.detach().cpu().numpy()]
            clf._random_state = 42

            jp = os.path.join(output_dir, 'model.joblib')
            joblib.dump(clf, jp)

            y_pred = clf.predict(val_df[feature_cols].values)
            y_true = val_df[label_column].values.astype(int)
            metrics = {
                'val_auc': float(auc),
                'accuracy': float(accuracy_score(y_true, y_pred)),
                'precision': float(precision_score(y_true, y_pred, zero_division=0)),
                'recall': float(recall_score(y_true, y_pred, zero_division=0)),
                'f1_score': float(f1_score(y_true, y_pred, zero_division=0)),
                'confusion_matrix': confusion_matrix(y_true, y_pred).tolist(),
                'epochs': num_epochs, 'train_rows': len(train_df), 'val_rows': len(val_df), 'namespace': ns,
            }
            mp = os.path.join(output_dir, 'metrics.json')
            with open(mp, 'w') as f:
                json.dump(metrics, f, indent=2)
            print(f'Metrics: AUC={auc:.4f} Acc={metrics["accuracy"]:.4f}')

            import boto3
            from botocore.client import Config as BC
            s3 = boto3.client('s3', endpoint_url=minio_endpoint, aws_access_key_id=minio_access_key,
                              aws_secret_access_key=minio_secret_key, config=BC(signature_version='s3v4'), region_name='us-east-1')
            try: s3.create_bucket(Bucket=minio_bucket)
            except Exception: pass
            s3.upload_file(pt_path, minio_bucket, f'{minio_model_prefix}/fraud_mlp_state_dict.pt')
            s3.upload_file(jp, minio_bucket, f'{minio_model_prefix}/model.joblib')
            s3.upload_file(mp, minio_bucket, f'{minio_model_prefix}/metrics.json')
            print(f'Uploaded to MinIO: s3://{minio_bucket}/{minio_model_prefix}/')

        if distributed:
            dist.barrier()
            dist.destroy_process_group()

    # ── Step 3: Submit the TrainJob ─────────────────────────────────────
    from kubeflow.trainer import CustomTrainer, TrainerClient

    trainer_client = TrainerClient()
    job_name = trainer_client.train(
        trainer=CustomTrainer(
            func=train_fraud_from_feast,
            func_args={
                'num_epochs': num_epochs, 'batch_size': batch_size, 'lr': learning_rate,
                'hidden_dim': hidden_dim, 'feast_start_date': feast_start_date,
                'feast_end_date': feast_end_date, 'label_column': 'fraud',
                'val_split': val_split, 'output_dir': '/tmp/model_output',
                'minio_endpoint': minio_endpoint, 'minio_access_key': minio_access_key,
                'minio_secret_key': minio_secret_key, 'minio_bucket': minio_bucket,
                'minio_model_prefix': minio_model_prefix, 'workshop_ns': workshop_ns,
            },
            num_nodes=1,
            resources_per_node={'cpu': '2', 'memory': '4Gi'},
            packages_to_install=[
                'torch', 'pandas', 'scikit-learn==1.5.1', 'pyarrow',
                'boto3', 'joblib', 'feast[postgres,grpc]', 'psycopg2-binary', 'grpcio',
            ],
        ),
        runtime='torch-distributed',
    )
    print(f'Submitted TrainJob: {job_name}')

    # ── Step 4: Wait for completion ─────────────────────────────────────
    trainer_client.wait_for_job_status(name=job_name, status={'Running'}, timeout=600)
    print(f'{job_name} is running. Streaming logs:')
    for line in trainer_client.get_job_logs(job_name, follow=True):
        print(line, end='')
    trainer_client.wait_for_job_status(name=job_name, timeout=120)
    print('TrainJob completed.')

print('Component 1 defined: submit_train_job')

Component 1 defined: submit_train_job


### Component 2 — Quality Gate

Reads `metrics.json` from MinIO (uploaded by the training job) and checks
whether the model's AUC meets the minimum threshold. If not, the pipeline fails
and KServe deployment is skipped.

In [4]:
@dsl.component(
    base_image='registry.access.redhat.com/ubi9/python-311:latest',
    packages_to_install=['boto3'],
)
def quality_gate(
    minio_endpoint: str,
    minio_access_key: str,
    minio_secret_key: str,
    minio_bucket: str,
    minio_model_prefix: str,
    min_auc: float,
):
    """Read metrics.json from MinIO, enforce quality threshold."""
    import json
    import boto3
    from botocore.client import Config

    s3 = boto3.client(
        's3', endpoint_url=minio_endpoint,
        aws_access_key_id=minio_access_key,
        aws_secret_access_key=minio_secret_key,
        config=Config(signature_version='s3v4'),
        region_name='us-east-1',
    )
    s3.download_file(minio_bucket, f'{minio_model_prefix}/metrics.json', '/tmp/metrics.json')

    with open('/tmp/metrics.json') as f:
        metrics = json.load(f)

    auc = metrics['val_auc']
    print('=' * 50)
    print('MODEL QUALITY GATE')
    print('=' * 50)
    print(f'  AUC:       {auc:.4f}  (threshold: >= {min_auc})')
    for k in ['accuracy', 'precision', 'recall', 'f1_score']:
        if k in metrics:
            print(f'  {k:10s}: {metrics[k]:.4f}')
    if 'confusion_matrix' in metrics:
        cm = metrics['confusion_matrix']
        print(f'  Confusion: TN={cm[0][0]} FP={cm[0][1]} FN={cm[1][0]} TP={cm[1][1]}')
    print('-' * 50)

    if auc >= min_auc:
        print(f'  PASSED — AUC {auc:.4f} >= {min_auc}')
    else:
        print(f'  FAILED — AUC {auc:.4f} < {min_auc}')
        raise RuntimeError(f'Quality gate failed: AUC={auc:.4f} < {min_auc}')
    print('=' * 50)

print('Component 2 defined: quality_gate')

Component 2 defined: quality_gate


### Component 3 — Deploy KServe InferenceService

Creates or patches a KServe `InferenceService` that reads the model from MinIO.
Uses the Kubernetes Python client directly — no KServe SDK needed.

In [5]:
@dsl.component(
    base_image='registry.access.redhat.com/ubi9/python-311:latest',
    packages_to_install=['kubernetes'],
)
def deploy_kserve(
    minio_bucket: str,
    minio_model_prefix: str,
    kserve_namespace: str,
    inference_service_name: str,
):
    """Create or patch a KServe InferenceService."""
    from kubernetes import client, config

    model_uri = f's3://{minio_bucket}/{minio_model_prefix}'
    print(f'Deploying InferenceService {inference_service_name} in {kserve_namespace}')
    print(f'Storage URI: {model_uri}')

    try:
        config.load_incluster_config()
    except Exception:
        config.load_kube_config()

    api = client.CustomObjectsApi()
    isvc = {
        'apiVersion': 'serving.kserve.io/v1beta1',
        'kind': 'InferenceService',
        'metadata': {'name': inference_service_name, 'namespace': kserve_namespace},
        'spec': {
            'predictor': {
                'serviceAccountName': 'kserve-minio-sa',
                'model': {
                    'modelFormat': {'name': 'sklearn'},
                    'storageUri': model_uri,
                    'resources': {
                        'requests': {'memory': '512Mi', 'cpu': '250m'},
                        'limits': {'memory': '1Gi', 'cpu': '500m'},
                    },
                },
            },
        },
    }

    try:
        api.get_namespaced_custom_object(
            group='serving.kserve.io', version='v1beta1',
            namespace=kserve_namespace, plural='inferenceservices',
            name=inference_service_name,
        )
        api.patch_namespaced_custom_object(
            group='serving.kserve.io', version='v1beta1',
            namespace=kserve_namespace, plural='inferenceservices',
            name=inference_service_name, body=isvc,
        )
        print('Patched existing InferenceService')
    except client.exceptions.ApiException as e:
        if e.status == 404:
            api.create_namespaced_custom_object(
                group='serving.kserve.io', version='v1beta1',
                namespace=kserve_namespace, plural='inferenceservices',
                body=isvc,
            )
            print('Created InferenceService')
        else:
            raise

print('Component 3 defined: deploy_kserve')

Component 3 defined: deploy_kserve


---
## 4. Pipeline Definition

Chains the three components:
1. **Train** → submits TrainerV2 job, waits for completion
2. **Quality Gate** → reads metrics, fails if AUC < threshold
3. **Deploy** → creates/patches KServe InferenceService (only if quality gate passes)

In [6]:
@dsl.pipeline(
    name='Fraud Detection — TrainerV2 + Feast + KServe',
    description='Submits a TrainerV2 TrainJob (Feast fetch, PyTorch train, MinIO upload), '
                'then quality gate, then KServe deploy.',
)
def fraud_pipeline(
    workshop_ns: str = 'mlops-workshop',
    feast_start_date: str = '2025-01-01',
    feast_end_date: str = '2025-03-31',
    num_epochs: int = 5,
    batch_size: int = 256,
    learning_rate: float = 0.001,
    hidden_dim: int = 64,
    val_split: float = 0.2,
    minio_endpoint: str = 'http://minio-service.kubeflow.svc.cluster.local:9000',
    minio_access_key: str = 'minio',
    minio_secret_key: str = 'minio123',
    minio_bucket: str = 'models',
    minio_model_prefix: str = 'fraud-detector',
    min_auc: float = 0.7,
    deploy_model: bool = True,
    inference_service_name: str = 'fraud-detector',
):
    train_task = submit_train_job(
        workshop_ns=workshop_ns,
        feast_start_date=feast_start_date,
        feast_end_date=feast_end_date,
        num_epochs=num_epochs,
        batch_size=batch_size,
        learning_rate=learning_rate,
        hidden_dim=hidden_dim,
        val_split=val_split,
        minio_endpoint=minio_endpoint,
        minio_access_key=minio_access_key,
        minio_secret_key=minio_secret_key,
        minio_bucket=minio_bucket,
        minio_model_prefix=minio_model_prefix,
    ).set_display_name('1. Submit TrainerV2 TrainJob')

    gate_task = quality_gate(
        minio_endpoint=minio_endpoint,
        minio_access_key=minio_access_key,
        minio_secret_key=minio_secret_key,
        minio_bucket=minio_bucket,
        minio_model_prefix=minio_model_prefix,
        min_auc=min_auc,
    ).after(train_task).set_display_name('2. Quality Gate')

    with dsl.If(deploy_model == True, name='Deploy if Approved'):
        deploy_kserve(
            minio_bucket=minio_bucket,
            minio_model_prefix=minio_model_prefix,
            kserve_namespace=workshop_ns,
            inference_service_name=inference_service_name,
        ).after(gate_task).set_display_name('3. Deploy KServe')

print('Pipeline defined.')

Pipeline defined.


---
## 5. Compile Pipeline

Compiles the pipeline to a YAML file that KFP can execute.

In [7]:
YAML_PATH = 'fraud_pipeline.yaml'
compiler.Compiler().compile(pipeline_func=fraud_pipeline, package_path=YAML_PATH)
print(f'Compiled → {YAML_PATH}')

Compiled → fraud_pipeline.yaml


---
## 6. Submit Pipeline Run

Connect to the KFP UI and submit the pipeline with your parameters.

**Before running:** update `KFP_ENDPOINT` to match your cluster's KFP Route URL.

In [8]:
import os
import kfp

KFP_ENDPOINT = 'http://kfp-ui-kubeflow.apps.rosa.k2s7v9j3f3g5j9l.yqif.p3.openshiftapps.com'
MY_NS = os.environ.get('WORKSHOP_NS', 'mlops-workshop')

kfp_client = kfp.Client(host=KFP_ENDPOINT)
print(f'Connected to KFP at {KFP_ENDPOINT}')
print(f'Workshop namespace: {MY_NS}')

Connected to KFP at http://kfp-ui-kubeflow.apps.rosa.k2s7v9j3f3g5j9l.yqif.p3.openshiftapps.com
Workshop namespace: mlops-workshop


/opt/app-root/lib64/python3.11/site-packages/kfp/client/client.py:159: FutureWarning: This client only works with Kubeflow Pipeline v2.0.0-beta.2 and later versions.
  warnings.warn(


In [9]:
PIPELINE_PARAMS = {
    'workshop_ns': MY_NS,
    'feast_start_date': '2025-01-01',
    'feast_end_date': '2025-03-31',
    'num_epochs': 5,
    'batch_size': 256,
    'learning_rate': 0.001,
    'hidden_dim': 64,
    'val_split': 0.2,
    'minio_endpoint': 'http://minio-service.kubeflow.svc.cluster.local:9000',
    'minio_access_key': 'minio',
    'minio_secret_key': 'minio123',
    'minio_bucket': 'models',
    'minio_model_prefix': f'fraud-detector-{MY_NS}',
    'min_auc': 0.7,
    'deploy_model': True,
    'inference_service_name': 'fraud-detector',
}

print('Pipeline parameters:')
for k, v in PIPELINE_PARAMS.items():
    print(f'  {k:30s} = {v}')

Pipeline parameters:
  workshop_ns                    = mlops-workshop
  feast_start_date               = 2025-01-01
  feast_end_date                 = 2025-03-31
  num_epochs                     = 5
  batch_size                     = 256
  learning_rate                  = 0.001
  hidden_dim                     = 64
  val_split                      = 0.2
  minio_endpoint                 = http://minio-service.kubeflow.svc.cluster.local:9000
  minio_access_key               = minio
  minio_secret_key               = minio123
  minio_bucket                   = models
  minio_model_prefix             = fraud-detector-mlops-workshop
  min_auc                        = 0.7
  deploy_model                   = True
  inference_service_name         = fraud-detector


In [10]:
run = kfp_client.create_run_from_pipeline_package(
    pipeline_file=YAML_PATH,
    arguments=PIPELINE_PARAMS,
    run_name='fraud-trainerv2-run',
    experiment_name='fraud-detection-trainerv2',
)
print(f'Run submitted: {run.run_id}')
print(f'View in UI: {KFP_ENDPOINT}/#/runs/details/{run.run_id}')

Run submitted: 77b1a667-b5db-40f8-bd9a-6b9f3c0a6cfc
View in UI: http://kfp-ui-kubeflow.apps.rosa.k2s7v9j3f3g5j9l.yqif.p3.openshiftapps.com/#/runs/details/77b1a667-b5db-40f8-bd9a-6b9f3c0a6cfc


---
## 7. Monitor Run

Wait for the pipeline to complete. This typically takes 5-8 minutes:
- ~2 min: package installation in training pod
- ~2 min: Feast fetch + training
- ~1 min: quality gate + KServe deploy

In [ ]:
kfp_client.wait_for_run_completion(run_id=run.run_id, timeout=900)
print('Pipeline run completed.')